In [1]:
import numpy as np
import pandas as pd
import os
import re
from sklearn.base import clone, BaseEstimator, RegressorMixin
from sklearn.metrics import cohen_kappa_score, accuracy_score, mean_squared_error
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
from scipy.optimize import minimize
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import optuna
from optuna.samplers import TPESampler

from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
# from keras.models import Model
# from keras.layers import Input, Dense
# from keras.optimizers import Adam
import torch
import torch.nn as nn
import torch.optim as optim
#from pytorch_tabnet.tab_model import TabNetRegressor
#from pytorch_tabnet.callbacks import Callback

from colorama import Fore, Style
from IPython.display import clear_output
import warnings
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor, XGBClassifier
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import cohen_kappa_score
#from pytorch_tabnet.metrics import Metric
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

SEED = 42
n_splits = 5

c:\Users\USER\anaconda3\envs\kaggle\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def calculate_regularity_index(df):
    """Calculate the regularity index based on light measurements."""
    start_date = pd.to_datetime('2024-01-01')

    df['timestep'] = start_date + pd.to_timedelta(df['relative_date_PCIAT'], unit='D')
    df['timestep'] += pd.to_timedelta(df['time_of_day'], unit='ns')
    df = df.set_index('timestep')

    # Filter for wearing times
    wear_df = df[df['non-wear_flag'] == 0.0]
    wear_df = wear_df.resample('5T').mean()

    # Grouping by hour and calculating daily pattern
    wear_df['hour'] = wear_df.index.hour + wear_df.index.minute / 60
    daily_pattern = wear_df.groupby('hour')['light'].mean()

    wear_df['date_only'] = wear_df.index.date
    daily_variation = wear_df.groupby(['date_only', 'hour'])['light'].mean().unstack()

    # 각 시간대별 데이터 포인트 수 계산
    counts_per_time_of_day = daily_variation.count()

    # 각 시간대에서의 일별 표준편차 계산 (결측치는 자동으로 제외됨)
    std_dev_per_time_of_day = daily_variation.std()

    # 규칙성 지표 계산
    # 표준편차를 데이터 포인트 수로 가중 평균하여 규칙성 지표 계산
    # 데이터 포인트 수가 많을수록 표준편차의 신뢰도가 높아지므로 이를 반영
    weighted_std = std_dev_per_time_of_day * counts_per_time_of_day
    regularity_index = 1 / (1 + weighted_std.sum())
        
    return regularity_index

def process_file(filename, dirname):
    df = pd.read_parquet(os.path.join(dirname, filename, 'part-0.parquet'))
    df = df[df['non-wear_flag']==0]
    #df.drop('step', axis=1, inplace=True)
    df = df[['enmo', 'anglez', 'light', 'non-wear_flag', 'time_of_day', 'relative_date_PCIAT']]
    regularity_index = calculate_regularity_index(df)
    df = df[['enmo', 'anglez', 'light']]
    return df.describe().values.reshape(-1)+regularity_index, filename.split('=')[1]

def load_time_series(dirname) -> pd.DataFrame:
    ids = os.listdir(dirname)
    
    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(lambda fname: process_file(fname, dirname), ids), total=len(ids)))
    
    stats, indexes = zip(*results)
    
    df = pd.DataFrame(stats, columns=[f"stat_{i}" for i in range(len(stats[0]))])
    df['id'] = indexes
    return df

class AutoEncoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dim*3),
            nn.ReLU(),
            nn.Linear(encoding_dim*3, encoding_dim*2),
            nn.ReLU(),
            nn.Linear(encoding_dim*2, encoding_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, input_dim*2),
            nn.ReLU(),
            nn.Linear(input_dim*2, input_dim*3),
            nn.ReLU(),
            nn.Linear(input_dim*3, input_dim),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
def perform_autoencoder(df, encoding_dim=5, epochs=50, batch_size=32):
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df)
    
    data_tensor = torch.FloatTensor(df_scaled)
    
    input_dim = data_tensor.shape[1]
    autoencoder = AutoEncoder(input_dim, encoding_dim)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(autoencoder.parameters())
    
    for epoch in range(epochs):
        for i in range(0, len(data_tensor), batch_size):
            batch = data_tensor[i : i + batch_size]
            optimizer.zero_grad()
            reconstructed = autoencoder(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()
            
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}]')
                 
    with torch.no_grad():
        encoded_data = autoencoder.encoder(data_tensor).numpy()
        
    df_encoded = pd.DataFrame(encoded_data, columns=[f'Enc_{i + 1}' for i in range(encoded_data.shape[1])])
    
    return df_encoded

In [ ]:
def data_preprocessing(df):
    PAQ_Total = []
    for i in range(len(df)):
        if pd.isnull(df.loc[i]['PAQ_A-PAQ_A_Total']):
            if pd.isnull(df.loc[i]['PAQ_C-PAQ_C_Total']):
                PAQ_Total.append(np.nan)
            else:
                PAQ_Total.append(df.loc[i]['PAQ_C-PAQ_C_Total'])
        else:
            PAQ_Total.append(df.loc[i]['PAQ_A-PAQ_A_Total'])
    df['PAQ_Total'] = PAQ_Total
    df = df.drop(['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total'], axis =1)
    df = df.drop(['Physical-Waist_Circumference','Fitness_Endurance-Time_Sec','Fitness_Endurance-Max_Stage'], axis =1 ) #no meaning + high variance column drop

    df = df.drop(['FGC-FGC_GSND','FGC-FGC_GSND_Zone', 'FGC-FGC_GSD', 'FGC-FGC_GSD_Zone'], axis=1) #GSND, GSD column drop
    df['Physical-Weight'] = df['Physical-Weight'].replace(0, np.nan)
    df['CGAS-CGAS_Score'] = df['CGAS-CGAS_Score'].replace(999.0, 99.0)
    df['Physical-BMI'] = df['Physical-BMI'].replace(0, np.nan)
    df['Physical-Height'] = df['Physical-Height'].replace(0, np.nan)
    df['Physical-Weight'] = df['Physical-Weight'].replace(0, np.nan)
    df['Physical-Diastolic_BP'] = df['Physical-Diastolic_BP'].replace(0, np.nan)
    df['Physical-Systolic_BP'] = df['Physical-Systolic_BP'].replace(0, np.nan)

    df['CGAS-CGAS_Score'] = df['CGAS-CGAS_Score'].replace(999.0, 99.0)
    df['Physical-BMI'] = df['Physical-BMI'].replace(0, np.nan)
    df['Physical-Height'] = df['Physical-Height'].replace(0, np.nan)
    df['Physical-Weight'] = df['Physical-Weight'].replace(0, np.nan)
    df['Physical-Diastolic_BP'] = df['Physical-Diastolic_BP'].replace(0, np.nan)
    df['Physical-Systolic_BP'] = df['Physical-Systolic_BP'].replace(0, np.nan)    
    
    
    thresholds = {
        'BIA-BIA_BMC': 400,
        'BIA-BIA_BMR': 10000,
        'BIA-BIA_DEE': 9000,
        'BIA-BIA_ECW': 150,
        'BIA-BIA_FFM': 400,
        'BIA-BIA_FFMI': 100,
        'BIA-BIA_FMI': -100,
        'BIA-BIA_Fat': -1000,
        'BIA-BIA_ICW': 200,
        'BIA-BIA_LDM': 100,
        'BIA-BIA_LST': 400,
        'BIA-BIA_SMM': 300,
        'BIA-BIA_TBW': 300
    }

    for column, threshold in thresholds.items():
        condition = df[column] > threshold if threshold > 0 else df[column] < threshold
        df[column] = np.where(condition, np.nan, df[column])
    
    return df
    
def feature_engineering(autoencoder, df, scaler):    
    season_cols = [col for col in df.columns if 'Season' in col]
    df = df.drop(season_cols, axis=1)
    BIA_cols = [col for col in df.columns if 'BIA' in col]
    df_BIA_encoded = perform_autoencoder_BIA(autoencoder, df, scaler)
    df = df.drop(BIA_cols, axis=1)
    for column in df_BIA_encoded:
        df[column] = df_BIA_encoded[column]
        
    df['BMI_Age'] = df['Physical-BMI'] * df['Basic_Demos-Age']
    df['Internet_Hours_Age'] = df['PreInt_EduHx-computerinternet_hoursday'] * df['Basic_Demos-Age']
    df['BMI_Internet_Hours'] = df['Physical-BMI'] * df['PreInt_EduHx-computerinternet_hoursday']
    #df['BFP_BMI'] = df['BIA-BIA_Fat'] / df['BIA-BIA_BMI']
    #df['FFMI_BFP'] = df['BIA-BIA_FFMI'] / df['BIA-BIA_Fat']
    #df['FMI_BFP'] = df['BIA-BIA_FMI'] / df['BIA-BIA_Fat']
    #df['LST_TBW'] = df['BIA-BIA_LST'] / df['BIA-BIA_TBW']
    #df['BFP_BMR'] = df['BIA-BIA_Fat'] * df['BIA-BIA_BMR']
    #df['BFP_DEE'] = df['BIA-BIA_Fat'] * df['BIA-BIA_DEE']
    #df['BMR_Weight'] = df['BIA-BIA_BMR'] / df['Physical-Weight']
    #df['DEE_Weight'] = df['BIA-BIA_DEE'] / df['Physical-Weight']
    #df['SMM_Height'] = df['BIA-BIA_SMM'] / df['Physical-Height']
    #df['Muscle_to_Fat'] = df['BIA-BIA_SMM'] / df['BIA-BIA_FMI']
    #df['Hydration_Status'] = df['BIA-BIA_TBW'] / df['Physical-Weight']
    #df['ICW_TBW'] = df['BIA-BIA_ICW'] / df['BIA-BIA_TBW']
    
    return df

In [ ]:
train = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/train.csv')
test = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/test.csv')
sample = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/sample_submission.csv')

In [ ]:
train_ts = load_time_series("/kaggle/input/child-mind-institute-problematic-internet-use/series_train.parquet")
test_ts = load_time_series("/kaggle/input/child-mind-institute-problematic-internet-use/series_test.parquet")

df_train = train_ts.drop('id', axis=1)
df_test = test_ts.drop('id', axis=1)

train_ts_encoded = perform_autoencoder(df_train, encoding_dim=5, epochs=100, batch_size=32)
test_ts_encoded = perform_autoencoder(df_test, encoding_dim=5, epochs=100, batch_size=32)

In [ ]:
def train_autoencoder_BIA(df, encoding_dim=2, epochs=50, batch_size=32):
    BIA_cols = [col for col in df.columns if ('BIA' in col) and ('Season' not in col)]
    clean_df = df[df[BIA_cols].notnull().all(axis=1)]
    clean_df = clean_df[BIA_cols]

    scaler = MinMaxScaler()
    df_scaled = scaler.fit_transform(clean_df)
    data_tensor = torch.FloatTensor(df_scaled)
    
    input_dim = data_tensor.shape[1]
    autoencoder = AutoEncoder(input_dim, encoding_dim)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(autoencoder.parameters())
    
    losses = []
    for epoch in range(epochs):
        for i in range(0, len(data_tensor), batch_size):
            batch = data_tensor[i : i + batch_size]
            optimizer.zero_grad()
            reconstructed = autoencoder(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()
        losses.append(loss.item())    
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}]')
                 
    
    return autoencoder, scaler, losses

def perform_autoencoder_BIA(autoencoder, df, scaler):
    BIA_cols = [col for col in df.columns if 'BIA' in col]
    df_BIA = df[BIA_cols]
    df_scaled = scaler.transform(df_BIA)
    data_tensor = torch.FloatTensor(df_scaled)

    with torch.no_grad():
        encoded_data = autoencoder.encoder(data_tensor).numpy()

    df_encoded = pd.DataFrame(encoded_data, columns=[f'Enc_B{i + 1}' for i in range(encoded_data.shape[1])])
    return df_encoded

In [ ]:
time_series_cols = train_ts_encoded.columns.tolist()
train_ts_encoded["id"]=train_ts["id"]
test_ts_encoded['id']=test_ts["id"]

train = pd.merge(train, train_ts_encoded, how="left", on='id')
test = pd.merge(test, test_ts_encoded, how="left", on='id')

# 모순 데이터 제거
kid_perfect = train[train['PCIAT-PCIAT_Total']==0]
no_perfect = kid_perfect[kid_perfect['PreInt_EduHx-computerinternet_hoursday']>2].index.tolist()

train = train.drop(index= no_perfect)
train = train.reset_index(drop=True)
train = data_preprocessing(train)
test = data_preprocessing(test)

In [ ]:
def KNN_Imputer(df, sii_or_not):
    if sii_or_not:
        imputer_columns = ['Basic_Demos-Age', 'Basic_Demos-Sex','CGAS-CGAS_Score','Physical-BMI',
                  'Physical-Height', 'Physical-Weight', 'Physical-Diastolic_BP','Physical-HeartRate', 'Physical-Systolic_BP',
                  'Fitness_Endurance-Time_Mins','FGC-FGC_CU', 'FGC-FGC_CU_Zone', 'FGC-FGC_PU', 'FGC-FGC_PU_Zone',
                   'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone', 'FGC-FGC_SRR', 'FGC-FGC_SRR_Zone','FGC-FGC_TL', 'FGC-FGC_TL_Zone', 'SDS-SDS_Total_Raw',
                   'SDS-SDS_Total_T','PreInt_EduHx-computerinternet_hoursday','PAQ_Total','BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMI',
                   'BIA-BIA_BMR', 'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_FFM','BIA-BIA_FFMI', 'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num',
                   'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM','BIA-BIA_TBW','sii']
    else:
        imputer_columns = ['Basic_Demos-Age', 'Basic_Demos-Sex','CGAS-CGAS_Score','Physical-BMI',
                  'Physical-Height', 'Physical-Weight', 'Physical-Diastolic_BP','Physical-HeartRate', 'Physical-Systolic_BP',
                  'Fitness_Endurance-Time_Mins','FGC-FGC_CU', 'FGC-FGC_CU_Zone', 'FGC-FGC_PU', 'FGC-FGC_PU_Zone',
                   'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone', 'FGC-FGC_SRR', 'FGC-FGC_SRR_Zone','FGC-FGC_TL', 'FGC-FGC_TL_Zone', 'SDS-SDS_Total_Raw',
                   'SDS-SDS_Total_T','PreInt_EduHx-computerinternet_hoursday','PAQ_Total','BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMI',
                   'BIA-BIA_BMR', 'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_FFM','BIA-BIA_FFMI', 'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num',
                   'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM','BIA-BIA_TBW']
    
    imputer = KNNImputer(n_neighbors=5)
    imputed_data = imputer.fit_transform(df[imputer_columns])
    train_imputed = pd.DataFrame(imputed_data, columns=imputer_columns)
    for col in df.columns:
        if col not in imputer_columns:
            train_imputed[col] = df[col]
        
    return train_imputed    

In [ ]:
BIA_encoder, BIA_scaler, losses = train_autoencoder_BIA(train)
train_f = KNN_Imputer(train, False) #sii빼고 impute
train_f = feature_engineering(BIA_encoder, train_f, BIA_scaler)
train_f = train_f.dropna(thresh=10, axis=0)
test = feature_engineering(BIA_encoder, test, BIA_scaler)

In [ ]:
featuresCols = ['Basic_Demos-Age', 'Basic_Demos-Sex',
                'CGAS-CGAS_Score', 'Physical-BMI',
                'Physical-Height', 'Physical-Weight', 
                'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP',
                'Fitness_Endurance-Time_Mins', 
                'FGC-FGC_CU', 'FGC-FGC_CU_Zone', 'FGC-FGC_PU',
                'FGC-FGC_PU_Zone', 'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone', 'FGC-FGC_SRR',
                'FGC-FGC_SRR_Zone', 'FGC-FGC_TL', 'FGC-FGC_TL_Zone', 'PAQ_Total', 'SDS-SDS_Total_Raw',
                'SDS-SDS_Total_T',
                'PreInt_EduHx-computerinternet_hoursday', 'sii','BMI_Age','Internet_Hours_Age','BMI_Internet_Hours',
               'Enc_B1', 'Enc_B2', 'Enc_1', 'Enc_2', 'Enc_3','Enc_4','Enc_5']

## 제외 목록
#'Physical-Waist_Circumference','Fitness_Endurance-Time_Sec','Fitness_Endurance-Max_Stage',
#'FGC-FGC_GSND', 'FGC-FGC_GSND_Zone', 'FGC-FGC_GSD', 'FGC-FGC_GSD_Zone'

#featuresCols += time_series_cols

train_f = train_f[featuresCols]

featuresCols = ['Basic_Demos-Age', 'Basic_Demos-Sex',
                'CGAS-CGAS_Score', 'Physical-BMI',
                'Physical-Height', 'Physical-Weight', 
                'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP',
                'Fitness_Endurance-Time_Mins', 
                'FGC-FGC_CU', 'FGC-FGC_CU_Zone', 'FGC-FGC_PU',
                'FGC-FGC_PU_Zone', 'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone', 'FGC-FGC_SRR',
                'FGC-FGC_SRR_Zone', 'FGC-FGC_TL', 'FGC-FGC_TL_Zone', 'PAQ_Total', 'SDS-SDS_Total_Raw',
                'SDS-SDS_Total_T',
                'PreInt_EduHx-computerinternet_hoursday', 'BMI_Age','Internet_Hours_Age','BMI_Internet_Hours',
               'Enc_B1', 'Enc_B2','Enc_1', 'Enc_2', 'Enc_3','Enc_4','Enc_5']

#featuresCols += time_series_cols
test = test[featuresCols]

#if np.any(np.isinf(train)):
#    train = train.replace([np.inf, -np.inf], np.nan)

In [ ]:
train_with_sii = train_f[train_f['sii'].notna()]
train_without_sii = train_f[train_f['sii'].notna()]

train_with_ts_f = train_f[train_f['Enc_1'].notna()]
train_without_ts_f = train_f.drop(columns=[f'Enc_{i}' for i in range(1, 6)])

test_with_ts = test[test['Enc_1'].notna()]
test_without_ts = test.drop(columns=[f'Enc_{i}' for i in range(1,6)])

sample_with_ts = sample.loc[test_with_ts.index]
sample_without_ts = sample

In [ ]:
def quadratic_weighted_kappa(y_true, y_pred):
    return cohen_kappa_score(y_true.round(0).astype(int), y_pred.round(0).astype(int), weights='quadratic')

def threshold_Rounder(oof_non_rounded, thresholds):
    return np.where(oof_non_rounded < thresholds[0], 0,
                    np.where(oof_non_rounded < thresholds[1], 1,
                             np.where(oof_non_rounded < thresholds[2], 2, 3)))

def evaluate_predictions(thresholds, y_true, oof_non_rounded):
    rounded_p = threshold_Rounder(oof_non_rounded, thresholds)
    return -quadratic_weighted_kappa(y_true, rounded_p)

In [ ]:
def train_sequential_classification(df):
    models = []
    df['label_1'] = df['sii'].apply(lambda x: 1 if x > 0 else 0)
    df['label_2'] = df['sii'].apply(lambda x: 1 if x > 1 else 0)
    df['label_3'] = df['sii'].apply(lambda x: 1 if x > 2 else 0)
    df = df.drop(['Enc_1','Enc_2','Enc_3', 'Enc_4', 'Enc_5'], axis=1)
    X = df.drop(['sii','label_1','label_2','label_3'], axis=1)

    #-----model 1-----#
    y = df['label_1']
    
    model_1 = make_pipeline(StandardScaler(),SVC(kernel='linear'))
    #model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model_1.fit(X, y)
    models.append(model_1)

    #-----model 2 -----#
    X = df[df['sii']>0]
    y = X['label_2']
    X = X.drop(['sii','label_1','label_2', 'label_3'], axis=1)

    model_2 = make_pipeline(StandardScaler(),SVC(kernel='linear'))
    #model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model_2.fit(X, y)
    models.append(model_2)

    #-----model 3 -----#
    X = df[df['sii']>1]
    y = X['label_3']
    X = X.drop(['sii','label_1','label_2', 'label_3'], axis=1)

    model_3 = make_pipeline(StandardScaler(),SVC(kernel='linear'))
    #model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    model_3.fit(X, y)
    models.append(model_3)

    return models

def pseudo_sii_labeling(models, df):
    train_df = df.drop(['sii'], axis=1)
    for i in range(len(df)):
        if pd.isna(df.loc[i, 'sii']):  # 결측치 확인
            for sii_val, model in enumerate(models):
                if model.predict(train_df.loc[[i]]) == 0:  # DataFrame 형태로 전달
                    df.loc[i, 'sii'] = sii_val  # SII 값을 업데이트
                    break  # SII를 예측했으므로 더 이상 검사할 필요 없음
            else:
                # 모든 모델이 예측에 실패한 경우 SII를 3으로 설정
                df.loc[i, 'sii'] = 3
    return df

In [ ]:
models = train_sequential_classification(train_with_sii)
train_without_ts_f = pseudo_sii_labeling(models, train_without_ts_f)

In [ ]:
def TrainML_with_TS(df, model_class, test_data):
    X = df.drop(['sii'], axis=1)
    y = df['sii']

    SKF = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    train_S = []
    test_S = []
    
    oof_non_rounded = np.zeros(len(y), dtype=float) 
    oof_rounded = np.zeros(len(y), dtype=int) 
    test_preds = np.zeros((len(test_data), n_splits))

    #for fold, (train_idx, test_idx) in enumerate(tqdm(SKF.split(X, y), desc="Training Folds", total=n_splits)):
    for fold, (train_idx, test_idx) in enumerate(SKF.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[test_idx]

        model = clone(model_class)
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        oof_non_rounded[test_idx] = y_val_pred
        y_val_pred_rounded = y_val_pred.round(0).astype(int)
        oof_rounded[test_idx] = y_val_pred_rounded

        train_kappa = quadratic_weighted_kappa(y_train, y_train_pred.round(0).astype(int))
        val_kappa = quadratic_weighted_kappa(y_val, y_val_pred_rounded)

        train_S.append(train_kappa)
        test_S.append(val_kappa)
        
        test_preds[:, fold] = model.predict(test_data)
        
        #print(f"Fold {fold+1} - Train QWK: {train_kappa:.4f}, Validation QWK: {val_kappa:.4f}")
        #clear_output(wait=True)

    #print(f"Mean Train QWK --> {np.mean(train_S):.4f}")
    #print(f"Mean Validation QWK ---> {np.mean(test_S):.4f}")

    KappaOPtimizer = minimize(evaluate_predictions,
                              x0=[0.5, 1.5, 2.5], args=(y, oof_non_rounded), 
                              method='Nelder-Mead')
    assert KappaOPtimizer.success, "Optimization did not converge."
    
    oof_tuned = threshold_Rounder(oof_non_rounded, KappaOPtimizer.x)
    tKappa = quadratic_weighted_kappa(y, oof_tuned)

    #print(f"----> || Optimized QWK SCORE :: {Fore.CYAN}{Style.BRIGHT} {tKappa:.3f}{Style.RESET_ALL}")

    tpm = test_preds.mean(axis=1)
    tpTuned = threshold_Rounder(tpm, KappaOPtimizer.x)
    
    submission = pd.DataFrame({
        'id': sample_with_ts['id'],
        'sii': tpTuned
    })

    return submission, tKappa, model

In [ ]:
def TrainML_without_TS(df, model_class, test_data):
    if 'Enc_1' in df.columns:
        X = df.drop(['sii','Enc_1','Enc_2','Enc_3','Enc_4','Enc_5'], axis=1)        
    else:
        X = df.drop(['sii'], axis=1)    
        
    print(X.shape)
    y = df['sii']
    SKF = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    train_S = []
    test_S = []
    
    oof_non_rounded = np.zeros(len(y), dtype=float) 
    oof_rounded = np.zeros(len(y), dtype=int) 
    test_preds = np.zeros((len(test_data), n_splits))

    #for fold, (train_idx, test_idx) in enumerate(tqdm(SKF.split(X, y), desc="Training Folds", total=n_splits)):
    for fold, (train_idx, test_idx) in enumerate(SKF.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[test_idx]

        model = clone(model_class)
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        oof_non_rounded[test_idx] = y_val_pred
        y_val_pred_rounded = y_val_pred.round(0).astype(int)
        oof_rounded[test_idx] = y_val_pred_rounded

        train_kappa = quadratic_weighted_kappa(y_train, y_train_pred.round(0).astype(int))
        val_kappa = quadratic_weighted_kappa(y_val, y_val_pred_rounded)

        train_S.append(train_kappa)
        test_S.append(val_kappa)
        
        test_preds[:, fold] = model.predict(test_data)
        
        #print(f"Fold {fold+1} - Train QWK: {train_kappa:.4f}, Validation QWK: {val_kappa:.4f}")
        #clear_output(wait=True)

    #print(f"Mean Train QWK --> {np.mean(train_S):.4f}")
    #print(f"Mean Validation QWK ---> {np.mean(test_S):.4f}")

    KappaOPtimizer = minimize(evaluate_predictions,
                              x0=[0.5, 1.5, 2.5], args=(y, oof_non_rounded), 
                              method='Nelder-Mead')
    assert KappaOPtimizer.success, "Optimization did not converge."
    
    oof_tuned = threshold_Rounder(oof_non_rounded, KappaOPtimizer.x)
    tKappa = quadratic_weighted_kappa(y, oof_tuned)

    #print(f"----> || Optimized QWK SCORE :: {Fore.CYAN}{Style.BRIGHT} {tKappa:.3f}{Style.RESET_ALL}")

    tpm = test_preds.mean(axis=1)
    tpTuned = threshold_Rounder(tpm, KappaOPtimizer.x)
    
    submission = pd.DataFrame({
        'id': sample_without_ts['id'],
        'sii': tpTuned
    })

    return submission, tKappa, model

In [ ]:
LGBM_with_ts_Params = {'num_leaves': 1600, 'max_depth': 25, 'learning_rate': 0.01762952152473708, 'n_estimators': 1340, 'min_data_in_leaf': 20, 'subsample': 0.7602603970898999, 'reg_alpha': 2.4747161275654252, 'reg_lambda': 4.490231430673087e-05, 'feature_fraction': 0.5535966623553268, 'bagging_fraction': 0.8261227378877046}
#0.4514480408858603


XGB_with_ts_Params = {'max_depth': 11, 'n_estimators': 1410, 'learning_rate': 0.09579128650816048, 'reg_alpha': 1.4948449339881247, 'reg_lambda': 0.01948812461478295, 'colsample_bylevel': 0.5930769589342116, 'colsample_bytree': 0.6054902932414683, 'min_child_weight': 3.052194195798615, 'subsample': 0.5098024191171716}
#0.45019828097138603


CAT_with_ts_Params = {'iterations': 500, 'learning_rate': 0.11408623915604481, 'reg_lambda': 39.4707212659438, 'depth': 6, 'min_data_in_leaf': 7}
#0.46053127472489075


Light_w_ts = LGBMRegressor(**LGBM_with_ts_Params, random_state=SEED, verbose=-1)
XGB_w_ts = XGBRegressor(**XGB_with_ts_Params, random_state=SEED)
CatBoost_w_ts = CatBoostRegressor(**CAT_with_ts_Params, random_state=SEED, verbose=0)

voting_model_w_ts = VotingRegressor(estimators=[
    ('lightgbm_w_ts', Light_w_ts),
    ('xgboost_w_ts', XGB_w_ts),
    ('catboost_w_ts', CatBoost_w_ts)
])

Submission_Light_w_ts, qwk_score_Light_w_ts, model_trained_Light_w_ts = TrainML_with_TS(train_with_ts_f,Light_w_ts, test_with_ts)
Submission_XGB_w_ts, qwk_score_XGB_w_ts, model_trained_XGB_w_ts = TrainML_with_TS(train_with_ts_f,XGB_w_ts, test_with_ts)
Submission_Cat_w_ts, qwk_score_Cat_w_ts, model_trained_Cat_w_ts = TrainML_with_TS(train_with_ts_f,CatBoost_w_ts, test_with_ts)
Submission_w_ts, qwk_score_w_ts, model_trained_w_ts = TrainML_with_TS(train_with_ts_f,voting_model_w_ts, test_with_ts)

Submission_Light_w_ts = Submission_Light_w_ts.rename(columns={'sii': 'sii_Light'})
Submission_XGB_w_ts = Submission_XGB_w_ts.rename(columns={'sii': 'sii_XGB'})
Submission_Cat_w_ts = Submission_Cat_w_ts.rename(columns={'sii': 'sii_CatBoost'})

merged_df = Submission_Light_w_ts.merge(Submission_XGB_w_ts, on='id') \
                            .merge(Submission_Cat_w_ts, on='id') 

merged_df

In [ ]:
# 각 모델의 예측값 출력
light_pred = model_trained_Light_w_ts.predict(test_with_ts)
xgb_pred = model_trained_XGB_w_ts.predict(test_with_ts)
cat_pred = model_trained_Cat_w_ts.predict(test_with_ts)
voting_pred = model_trained_w_ts.predict(test_with_ts)

# DataFrame으로 정리해서 각 모델의 예측값과 VotingRegressor의 최종 예측값 비교
pred_df = pd.DataFrame({
    'id': Submission_w_ts['id'],
    'sii_Light': light_pred,
    'sii_XGB': xgb_pred,
    'sii_CatBoost': cat_pred,
    'sii_Voting': voting_pred
})

pred_df

In [ ]:
LGBM_without_ts_Params = {'num_leaves': 1300, 'max_depth': 19, 'learning_rate': 0.008557715394965188, 'n_estimators': 1190, 'min_data_in_leaf': 35, 'subsample': 0.8131721977885248, 'reg_alpha': 19.899989694664676, 'reg_lambda': 0.0003466760105686093, 'feature_fraction': 0.5974259919436959, 'bagging_fraction': 0.6272851710996933}
#0.5843980321096455


XGB_without_ts_Params = {'max_depth': 13, 'n_estimators': 900, 'learning_rate': 0.004423370060356562, 'reg_alpha': 0.0003047217358724957, 'reg_lambda': 0.00018291266736302026, 'colsample_bylevel': 0.890383013473116, 'colsample_bytree': 0.5420939563436876, 'min_child_weight': 8.779929138366331, 'subsample': 0.27560048505188195}
# 0.5827119395432285

CAT_without_ts_Params = {'iterations': 450, 'learning_rate': 0.12363907115850654, 'reg_lambda': 41.52066748426383, 'depth': 4, 'min_data_in_leaf': 18}
#0.5326374457703693

Light_wo_ts = LGBMRegressor(**LGBM_without_ts_Params, random_state=SEED, verbose=-1)
XGB_wo_ts = XGBRegressor(**XGB_without_ts_Params, random_state=SEED)
CatBoost_wo_ts = CatBoostRegressor(**CAT_without_ts_Params, random_state=SEED, verbose=0)

voting_model_wo_ts = VotingRegressor(estimators=[
    ('lightgbm_wo_ts', Light_wo_ts),
    ('xgboost_wo_ts', XGB_wo_ts),
    ('catboost_wo_ts', CatBoost_wo_ts),
    #('tabnet_wo_ts', TabNet_wo_ts),
])

Submission_Light_wo_ts, qwk_score_Light_wo_ts, model_trained_Light_wo_ts = TrainML_without_TS(train_without_ts_f,Light_wo_ts, test_without_ts)
Submission_XGB_wo_ts, qwk_score_XGB_wo_ts, model_trained_XGB_wo_ts = TrainML_without_TS(train_without_ts_f,XGB_wo_ts, test_without_ts)
Submission_Cat_wo_ts, qwk_score_Cat_wo_ts, model_trained_Cat_wo_ts = TrainML_without_TS(train_without_ts_f,CatBoost_wo_ts, test_without_ts)
Submission_wo_ts, qwk_score_wo_ts, model_trained_wo_ts = TrainML_without_TS(train_without_ts_f,voting_model_wo_ts, test_without_ts)

Submission_Light_wo_ts = Submission_Light_wo_ts.rename(columns={'sii': 'sii_Light'})
Submission_XGB_wo_ts = Submission_XGB_wo_ts.rename(columns={'sii': 'sii_XGB'})
Submission_Cat_wo_ts = Submission_Cat_wo_ts.rename(columns={'sii': 'sii_CatBoost'})

merged_df = Submission_Light_wo_ts.merge(Submission_XGB_wo_ts, on='id') \
                            .merge(Submission_Cat_wo_ts, on='id')

merged_df

In [ ]:
# 각 모델의 예측값 출력
light_pred = model_trained_Light_wo_ts.predict(test_without_ts)
xgb_pred = model_trained_XGB_wo_ts.predict(test_without_ts)
cat_pred = model_trained_Cat_wo_ts.predict(test_without_ts)
voting_pred = model_trained_wo_ts.predict(test_without_ts)

# DataFrame으로 정리해서 각 모델의 예측값과 VotingRegressor의 최종 예측값 비교
pred_df = pd.DataFrame({
    'id': Submission_wo_ts['id'],
    'sii_Light': light_pred,
    'sii_XGB': xgb_pred,
    'sii_CatBoost': cat_pred,
    'sii_Voting': voting_pred
})

pred_df

In [ ]:
Submission_wo_ts.update(Submission_w_ts)
Submission_wo_ts.to_csv('submission.csv', index=False)
Submission_wo_ts